# Observabilidade Multi-Agente com Amazon Bedrock AgentCore

## Visão Geral

Este tutorial demonstra como construir sistemas multi-agente com observabilidade completa usando Amazon Bedrock AgentCore Runtimes e CloudWatch GenAI Observability.

Exploraremos duas arquiteturas:
1. **Parte 1: Runtime Único** - Todos os agentes em um único runtime (mais simples, traces unificados)
2. **Parte 2: Multi-Runtime** - Agentes em runtimes separados com frameworks mistos (traces vinculados)

### Detalhes do Tutorial

| Informação | Detalhes |
|:------------|:--------|
| Tipo de tutorial | Conversacional |
| Tipo de agente | Multi-Agente (padrão Supervisor) |
| Frameworks Agênticos | Strands Agents & LangGraph |
| Modelo LLM | Anthropic Claude Haiku 4.5 |
| Funcionalidades Principais | Coordenação multi-agente, observabilidade OTEL |
| SDK utilizado | Amazon BedrockAgentCore Python SDK, boto3, Strands Agents, LangGraph |

### O Que Você Vai Aprender

- Como construir sistemas multi-agente com Strands e LangGraph
- Como a observabilidade funciona com `Strands Agents` e `LangGraph`
- Como usar OpenTelemetry baggage para propagação de sessão
- Como correlacionar traces entre múltiplos runtimes
- Como visualizar e analisar traces no painel GenAI Observability

---
**Duas arquiteturas:**
1. **Parte 1: Runtime Único** - Todos os agentes em um runtime (traces unificados)
2. **Parte 2: Multi-Runtime com múltiplos frameworks** - Agentes distribuídos (traces vinculados via correlação de sessão)

### Comparação

| Aspecto | Runtime Único | Multi-Runtime |
|--------|----------------|---------------------|
| **Arquitetura** | 1 runtime, todos os agentes | 3 runtimes separados |
| **Frameworks** | Todos Strands | Strands Agents + LangGraph |
| **Comunicação** | Chamadas de função diretas | Invocações de runtime com `runtimeSessionId` |
| **Propagação de Sessão** | Automática dentro do runtime | OpenTelemetry baggage |
| **IAM** | Role única | Roles separadas por agente |
| **Visualização de Trace** | Árvore de trace única e unificada | Traces correlacionados via session ID |
| **Complexidade** | Mais simples | Mais complexo |
| **Caso de Uso** | Time único, integração estreita | Times diferentes, frameworks diferentes |

> **Nota:** Este é um tutorial demonstrando **padrões de arquitetura multi-agente** e **configuração de observabilidade** para fins educacionais.

---

## Visão Geral da Arquitetura

```
                                REQUISIÇÃO DO USUÁRIO
                                     │
                 ┌───────────────────┴───────────────────┐
                 │                                       │
                 ▼                                       ▼
┌────────────────────────────────────┐  ┌────────────────────────────────────┐
│     PARTE 1: RUNTIME ÚNICO         │  │     PARTE 2: MULTI-RUNTIME         │
│     (Simples & Unificado)          │  │     (Distribuído & Escalável)      │
└────────────────────────────────────┘  └────────────────────────────────────┘

        RUNTIME ÚNICO                          MULTI-RUNTIME
┌──────────────────────────────┐       ┌──────────────────────────────┐
│   AGENTCORE RUNTIME ÚNICO    │       │   ORQUESTRADOR (runtime_with_strands_and_bedrock_models-pt.ipynb)     │
│                              │       │   AgentCore Runtime #1       │
│  ┌────────────────────────┐  │       │   [OTel Baggage: session.id] │
│  │  ORQUESTRADOR (runtime_with_strands_and_bedrock_models-pt.ipynb)│  │       └──────────────┬───────────────┘
│  │         │              │  │                      │
│  │    ┌────┴────┐         │  │                      |
│  │    ▼         ▼         │  │              ┌───────┴───────┐
│  │ VIAGEM    CLIMA        │  │              │               │
│  │(runtime_with_strands_and_bedrock_models-pt.ipynb) (runtime_with_strands_and_bedrock_models-pt.ipynb)     │  │              ▼               ▼
│  │web_search get_weather  │  │       ┌─────────────┐ ┌─────────────┐
│  └────────────────────────┘  │       │AGENTE VIAGEM│ │AGENTE CLIMA │
│                              │       │ (runtime_with_strands_and_bedrock_models-pt.ipynb)   │ │ (LangGraph) │
│  → Árvore de trace única     │       │ Runtime #2  │ │ Runtime #3  │
│    e unificada               │       │ web_search  │ │ get_weather │
└──────────────────────────────┘       └─────────────┘ └─────────────┘

        OBSERVABILIDADE                        OBSERVABILIDADE
┌──────────────────────────────┐       ┌──────────────────────────────┐
│  Árvore de Trace Única       │       │  Traces Correlacionados      │
│  e Unificada                 │       │  por Sessão                  │
│                              │       │                              │
│  Orquestrador                │       │                              │
│       │                      │       │  Trace do Orquestrador       │
│       ├── Agente de Viagem   │       │  Trace do Agente de Viagem   │
│       │    └── web_search    │       │  Trace do Agente de Clima    │
│       └── Agente de Clima    │       │                              │
│            └── get_weather   │       │                              │
└──────────────────────────────┘       └──────────────────────────────┘
                 │                                       │
                 └───────────────────┬───────────────────┘
                                     ▼
                    ┌────────────────────────────────────┐
                    │  CloudWatch GenAI Observability    │
                    └────────────────────────────────────┘
```

## Pré-requisitos

1. AWS CLI configurado (`aws configure`) com o conjunto mínimo de permissões necessárias.
2. Acesso ao modelo Amazon Bedrock para `global.anthropic.claude-haiku-4-5-20251001-v1:0`
3. CloudWatch Transaction Search (habilitado)[https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/Enable-TransactionSearch.html]

## Configuração

In [ ]:
!pip install -r requirements.txt --quiet

In [ ]:
import os
import boto3
import time
from pathlib import Path
from boto3.session import Session
from bedrock_agentcore_starter_toolkit import Runtime

# Set BASE_DIR as an absolute path that won't change
BASE_DIR = Path(os.getcwd()).resolve()
print(f"Base directory: {BASE_DIR}")

# Import utilities
import sys
sys.path.insert(0, str(BASE_DIR))
from utils import update_orchestrator_permissions, cleanup_runtime, cleanup_ssm_parameters

region = Session().region_name
print(f"Using region: {region}")

Helper para verificar o status do runtime até que o deploy seja concluído.

In [ ]:
def wait_for_ready(runtime):
    """Wait for runtime to be ready."""
    status = runtime.status().endpoint['status']
    print(status)
    while status not in ['READY', 'CREATE_FAILED', 'UPDATE_FAILED']:
        time.sleep(10)
        status = runtime.status().endpoint['status']
        print(f"Status: {status}")
    return status

---
# Parte 1: Multi-Agente em Runtime Único

Todos os agentes (Orquestrador, Viagem, Clima) rodam em **um único runtime** com chamadas de função diretas.

Nesta seção, fazemos o deploy de todos os agentes (Orquestrador, Viagem, Clima) em um **único AgentCore Runtime**.

**Arquitetura:**
```
┌─────────────────────────────────────┐
│     AgentCore Runtime Único         │
│  ┌─────────────────────────────┐    │
│  │    ORQUESTRADOR (runtime_with_strands_and_bedrock_models-pt.ipynb)   │    │
│  │         │                   │    │
│  │    ┌────┴────┐              │    │
│  │    ▼         ▼              │    │
│  │ VIAGEM    CLIMA             │    │
│  │ (runtime_with_strands_and_bedrock_models-pt.ipynb) (runtime_with_strands_and_bedrock_models-pt.ipynb)         │    │
│  │ web_search get_weather      │    │
│  └─────────────────────────────┘    │
│                                     │
│                                     │
│  → Árvore de trace única e          │
│    unificada                        │
└─────────────────────────────────────┘
```

- **Benefício**: Configuração simples, árvore de trace única e unificada no CloudWatch

Faça o deploy do runtime único. O starter toolkit cuida de:
- Construir a imagem Docker e enviar para o ECR
- Criar a IAM execution role com as permissões necessárias
- Fazer o deploy no AgentCore Runtime

In [ ]:
# Change to single_runtime directory
os.chdir(BASE_DIR / 'single_runtime')

single_runtime = Runtime()
single_runtime.configure(
    entrypoint="multi_agent.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="single_runtime_demo"
)

single_launch = single_runtime.launch()
print(f"Agent ARN: {single_launch.agent_arn}")

# Change back to base directory
os.chdir(BASE_DIR)

In [ ]:
wait_for_ready(single_runtime)

### Testar o Runtime Único

In [ ]:
# Invocation 1
print("=== Invocation 1 ===")
response = single_runtime.invoke({"prompt": "What's the weather in Paris and what should I visit?"})
print(response)





In [ ]:
# Invocation 2
print("=== Invocation 2 ===")
response = single_runtime.invoke({"prompt": "What's the weather in Tokyo and recommend some local food?"})
print(response)

### Visualizar os traces no Painel GenAI Observability do CloudWatch

<div style="text-align:left">
    <img src="images/single_runtime.png" width="50%"/>
</div>


---
# Parte 2: Multi-Runtime

Três runtimes separados se comunicando via `invoke_agent_runtime()`.

```
┌───────────────────────────────────────────────────────────┐
│                  ORQUESTRADOR (runtime_with_strands_and_bedrock_models-pt.ipynb)                   │
│                  AgentCore Runtime #1                     │
└─────────────────────────┬─────────────────────────────────┘
                          │ invoke_agent_runtime()
          ┌───────────────┴───────────────┐
          ▼                               ▼
┌───────────────────────┐     ┌───────────────────────┐
│   VIAGEM (runtime_with_strands_and_bedrock_models-pt.ipynb)    │     │  CLIMA (LangGraph)    │
│   Runtime #2          │     │  Runtime #3           │
│   web_search          │     │                       │
└───────────────────────┘     └───────────────────────┘
```

### 2.1 Deploy do Agente de Viagem (runtime_with_strands_and_bedrock_models-pt.ipynb) no AgentCore Runtime

In [ ]:
# Change to travel_agent directory
os.chdir(BASE_DIR / "travel_agent")

Armazenar o ARN do agente no SSM para descoberta pelo orquestrador e armazenamento seguro

In [ ]:
travel_runtime = Runtime()
travel_runtime.configure(
    entrypoint="main.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="travel_subagent_strands"
)

travel_launch = travel_runtime.launch()
travel_arn = travel_launch.agent_arn
print(f"Travel Agent ARN: {travel_arn}")

In [ ]:
ssm = boto3.client('ssm')
ssm.put_parameter(Name='/agents/travel_agent_arn', Value=travel_arn, Type='String', Overwrite=True)
ssm.put_parameter(Name='/agents/travel_agent_provider', Value='travel_agent_strands', Type='String', Overwrite=True)
print("Travel Agent metadata saved to SSM")

In [ ]:
wait_for_ready(travel_runtime)

print("\n" + "="*80)
print("📊 ENABLE OBSERVABILITY FOR TRAVEL AGENT RUNTIME")
print("="*80)
print("""
To enable vended logs and tracing for this runtime, follow these steps in the AWS Console and ensure you have Transaction Search enabled in your account:

🔹 CONFIGURE LOG DELIVERY:
1. Open the Agent Runtime page in the AgentCore console
2. In the Runtime agents pane, select the runtime: 'travel_subagent_strands'
3. Scroll down to the Log delivery pane and from Add drop-down, choose:
   - Amazon CloudWatch Logs (recommended)
4. Configure log delivery details:
   - Log type: APPLICATION_LOGS
   - Destination log group: /aws/vendedlogs/bedrock-agentcore/travel_subagent_strands
5. Choose Add
6. Verify that log delivery status shows "Delivery active"

🔹 CONFIGURE TRACING:
1. In the same runtime agent details page
2. In the Tracing pane, choose Edit
3. Toggle the widget to Enable
4. Choose Save
""")

# Change back to base directory
os.chdir(BASE_DIR)

<div style="text-align:left">
    <img src="images/vended_logs_trace_runtime.png" width="50%"/>
</div>

### 2.2 Deploy do Agente de Clima (LangGraph) no AgentCore Runtime


In [ ]:
# Change to weather_agent directory
os.chdir(BASE_DIR / "weather_agent")

weather_runtime = Runtime()
weather_runtime.configure(
    entrypoint="main.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="weather_subagent_lang"
)

weather_launch = weather_runtime.launch()
weather_arn = weather_launch.agent_arn
print(f"Weather Agent ARN: {weather_arn}")

In [ ]:
ssm.put_parameter(Name='/agents/weather_agent_arn', Value=weather_arn, Type='String', Overwrite=True)
ssm.put_parameter(Name='/agents/weather_agent_provider', Value='weather_agent_lang', Type='String', Overwrite=True)
print("Weather Agent metadata saved to SSM")

In [ ]:
wait_for_ready(weather_runtime)

print("\n" + "="*80)
print("📊 ENABLE OBSERVABILITY FOR WEATHER AGENT RUNTIME")
print("="*80)
print("""
To enable vended logs and tracing for this runtime, follow these steps in the AWS Console:

🔹 CONFIGURE LOG DELIVERY:
1. Open the Agent Runtime page in the AgentCore console
2. In the Runtime agents pane, select the runtime: 'weather_agent_lang'
3. Scroll down to the Log delivery pane and from Add drop-down, choose:
   - Amazon CloudWatch Logs (recommended)
4. Configure log delivery details:
   - Log type: APPLICATION_LOGS
   - Destination log group: /aws/vendedlogs/bedrock-agentcore/weather_agent_lang
5. Choose Add
6. Verify that log delivery status shows "Delivery active"

🔹 CONFIGURE TRACING:
1. In the same runtime agent details page
2. In the Tracing pane, choose Edit
3. Toggle the widget to Enable
4. Choose Save

""")

# Change back to base directory
os.chdir(BASE_DIR)

### 2.3 Deploy do Agente Orquestrador

Roteia consultas para sub-agentes via `invoke_agent_runtime()` com propagação de session ID.

In [ ]:
# Change to orchestrator_agent directory
os.chdir(BASE_DIR / "orchestrator_agent")

orchestrator_runtime = Runtime()
orchestrator_runtime.configure(
    entrypoint="main.py",
    auto_create_execution_role=True,
    auto_create_ecr=True,
    requirements_file="requirements.txt",
    region=region,
    agent_name="orchestrator_strands"
)

orchestrator_launch = orchestrator_runtime.launch()
print(f"Orchestrator ARN: {orchestrator_launch.agent_arn}")

In [ ]:
# Verify these values first
print(f"Travel ARN: {travel_arn}")
print(f"Weather ARN: {weather_arn}")
print(f"Orchestrator ID: {orchestrator_launch.agent_id}")

Adicionar permissões IAM para o orquestrador invocar sub-agentes.

**Importante**: Após atualizar as permissões IAM, aguardamos 60 segundos para que as alterações se propaguem pelos serviços AWS. Isso evita erros de "AccessDeniedException" quando o orquestrador tenta acessar parâmetros do SSM ou invocar sub-agentes.

In [ ]:
# Import utils from the base directory
import sys
sys.path.insert(0, str(BASE_DIR))
from utils import update_orchestrator_permissions

update_orchestrator_permissions(
    sub_agent_arns=[travel_arn, weather_arn],
    orchestrator_agent_id=orchestrator_launch.agent_id,
    region=region
)

# Wait for IAM propagation
print("⏳ Waiting 60 seconds for IAM permissions to propagate...")
print("   This ensures the orchestrator can access SSM parameters and invoke sub-agents.")
time.sleep(60)
print("✅ IAM propagation wait complete")

In [ ]:
wait_for_ready(orchestrator_runtime)

print("\n" + "="*80)
print("📊 ENABLE OBSERVABILITY FOR ORCHESTRATOR RUNTIME")
print("="*80)
print("""
To enable vended logs and tracing for this runtime, follow these steps in the AWS Console:

🔹 CONFIGURE LOG DELIVERY:
1. Open the Agent Runtime page in the AgentCore console
2. In the Runtime agents pane, select the runtime: 'orchestrator_strands'
3. Scroll down to the Log delivery pane and from Add drop-down, choose:
   - Amazon CloudWatch Logs (recommended)
4. Configure log delivery details:
   - Log type: APPLICATION_LOGS
   - Destination log group: /aws/vendedlogs/bedrock-agentcore/orchestrator_strands
5. Choose Add
6. Verify that log delivery status shows "Delivery active"

🔹 CONFIGURE TRACING:
1. In the same runtime agent details page
2. In the Tracing pane, choose Edit
3. Toggle the widget to Enable
4. Choose Save

""")

# Change back to base directory
os.chdir(BASE_DIR)

### Testar a Configuração Multi-Runtime e Multi-Framework

In [ ]:
# Pause to enable observability settings
print("⏸️  IMPORTANT: Before testing the multi-runtime system:")
print("1. Enable vended logs and tracing for all three runtimes in the AWS Console")
print("2. Follow the instructions printed above for each runtime")
print("3. Wait 1-2 minutes for settings to propagate")
print("\n✅ Once you've enabled observability for all runtimes, proceed to test the system")


### 📊 Resumo da Configuração de Observabilidade do AgentCore

Após habilitar vended logs e tracing para todos os três runtimes junto com Agent Observability, você terá visibilidade da saúde do sistema junto com o comportamento do seu agente:

| Runtime | Log Group | Tracing |
|---------|-----------|--------|
| Agente de Viagem | `/aws/vendedlogs/bedrock-agentcore/travel_subagent_strands` | Habilitado |
| Agente de Clima | `/aws/vendedlogs/bedrock-agentcore/weather_subagent_lang` | Habilitado |
| Orquestrador | `/aws/vendedlogs/bedrock-agentcore/orchestrator_strands` | Habilitado |

**Principais Benefícios:**
- **Vended Logs**: Logs da aplicação de cada runtime são automaticamente entregues ao CloudWatch
- **Distributed Tracing**: Traces são correlacionados entre runtimes
- **GenAI Observability**: Visibilidade completa das interações multi-agente

In [ ]:
# Invocation 1
print("=== Invocation 1 ===")
response = orchestrator_runtime.invoke({"prompt": "What's the weather in New York and what museums should I visit?"})
print(response)

print("\n" + "="*50 + "\n")

# Invocation 2
print("=== Invocation 2 ===")
response = orchestrator_runtime.invoke({"prompt": "What's the weather in Seattle and what are the best parks to visit?"})
print(response)

### Visualizar Traces

Para visualizar os traces e dados de observabilidade do seu sistema multi-agente:

## Observabilidade do AgentCore no Amazon CloudWatch

Após habilitar vended logs e tracing para todos os runtimes, siga estes passos para visualizar seus dados de observabilidade:


- Vended logs e tracing estão habilitados para todos os três runtimes
- Aguarde 2-3 minutos para que os traces apareçam após as invocações

### Visualizando Seu Sistema Multi-Agente no Painel GenAI Observability

#### 1. **Visão Geral do Bedrock AgentCore**
Navegue até **Console AWS** → **CloudWatch** → **GenAI Observability**

Você verá todos os seus agentes:
- `orchestrator_strands`
- `travel_subagent_strands`
- `weather_agent_lang`

Filtre os dados por período de tempo para focar nas suas invocações de teste recentes.

#### 2. **Métricas de Runtime (Todos os Agentes)**
No painel principal, visualize as métricas de runtime de todos os agentes:

<div style="text-align:left">
    <img src="images/runtime_metrics.png" width="50%"/>
</div>


#### 3. **Visão Por Agente**
Clique em qualquer agente específico (ex.: `orchestrator_strands`) para ver:
- Métricas de runtime específicas do agente
- Padrões de requisição/resposta
- Mudanças de desempenho
- Filtragem por período de tempo personalizado

<div style="text-align:left">
    <img src="images/per_agent.png" width="50%"/>
</div>

#### 4. **Visão de Sessões**
Navegue até a aba **Sessions View** para ver:
- Todas as sessões associadas a cada agente
- Session IDs que vinculam requisições entre múltiplos runtimes
- Fluxo de requisições do orquestrador para os sub-agentes

#### 5. **Visão de Traces**
Na aba **Trace View**, examine:
- Traces detalhados e informações de span para cada runtime em timelines
- A visão completa da trajetória
  ```
  Orquestrador (recebe requisição)
  ├── Invocar Agente de Viagem (com runtimeSessionId)
  │   └── Chamadas da ferramenta Web Search
  ├── Invocar Agente de Clima (com runtimeSessionId)
  │   └── Chamadas da ferramenta Weather
  └── Agregação de resposta
  ```
- Atributos de span mostrando:
  - Invocações de modelo
  - Execuções de ferramentas
  - Detalhamento de latência


<div style="text-align:left">
    <img src="images/full_trace.png" width="80%"/>
</div>


#### 6. **Traces Correlacionados Entre Runtimes**
Para visualizar a interação multi-agente completa:
1. Encontre um trace do orquestrador
2. Você verá traces correlacionados de todos os três runtimes mostrando a execução distribuída completa e poderá visualizar também para os sub-agentes.

### Principais Funcionalidades de Observabilidade para Sistemas Multi-Agente

| Funcionalidade | Descrição | Caso de Uso |
|---------|-------------|----------|
| **Vended Logs** | Logs da aplicação entregues automaticamente ao CloudWatch | Depurar lógica e erros do agente |
| **Distributed Tracing** | Traces correlacionados | Rastrear requisições entre agentes |
| **Correlação de Sessão** | Vincular todos os spans de uma única requisição do usuário | Visibilidade completa do fluxo da requisição |
| **Spans de Execução de Ferramentas** | Ver chamadas individuais de ferramentas (web_search, get_weather) | Transparência de desempenho |
| **Métricas de Invocação de Modelo** | Uso de tokens e latência por chamada de modelo | Monitoramento de custo e desempenho |

### Vended Logs e Tracing para AgentCore Runtime junto com o agente

Clique nas diversas funcionalidades do painel GenAI observability para explorar informações detalhadas de traces, métricas de desempenho e comportamento do sistema.

---
# Limpeza

Por favor, também limpe todos os recursos associados para evitar custos adicionais como ECR, logs, etc.

In [ ]:
# Clean up all resources
cleanup_runtime(single_launch, "single_runtime_demo", region)
cleanup_runtime(travel_launch, "travel_agent_strands", region)
cleanup_runtime(weather_launch, "weather_agent_lang", region)
cleanup_runtime(orchestrator_launch, "orchestrator_strands", region)

cleanup_ssm_parameters([
   '/agents/travel_agent_arn',
   '/agents/travel_agent_provider',
    '/agents/weather_agent_arn',
    '/agents/weather_agent_provider'
])

print("Cleanup complete!")